# 🎼 [키아노스 국립음악창작원] 캐글(Kaggle) 고성능 GPU 원격 연산 기지
**통제 기구:** 키아노스 국제질서유지 보안사령부 (K-SEF)  
**운용 목적:** 구글 캐글의 무료 듀얼 GPU(NVIDIA T4 x2) 자원을 합법적으로 징발하여 키아노스 고품질 AI 음원을 렌더링

---

### 1단계: 필수 AI 오디오 엔진 라이브러리 설치 (최초 1회 실행)

In [ ]:
!pip install -q transformers scipy torch torchaudio gradio pyngrok

### 2단계: GPU 연산 장치 확인 (Kaggle T4 x 2 연동)

In [ ]:
import torch

print("====================================================")
if torch.cuda.is_available():
    print(f"✅ K-SEF 보안 승인: GPU 활성화 성공!")
    print(f"• 장치명: {torch.cuda.get_device_name(0)}")
    print(f"• GPU 개수: {torch.cuda.device_count()}개")
    print(f"• 전용 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU가 활성화되지 않았습니다. 우측 사이드바 'Settings -> Accelerator'에서 'GPU T4 x2'를 선택하십시오.")
print("====================================================")

### 3단계: 고품질 AI 음악 생성 엔진 로드 (대형 MusicGen / ACE-Step 지원)

In [ ]:
import os
import time
import scipy.io.wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# 캐글 고성능 환경에서는 고음질 'medium' 모델을 기본 사용합니다.
MODEL_ID = "facebook/musicgen-medium"
print(f"[엔진 탑재] {MODEL_ID} 모델을 캐글 고속 VRAM에 적재합니다...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = MusicgenForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print("✨ [탑재 완료] 키아노스 고음질 음원 렌더링 준비 완료!")

### 4단계: 키아노스 전용 웹 UI 가동 (Gradio 기반 원클릭 작곡실)

In [ ]:
import gradio as gr

os.makedirs("outputs", exist_ok=True)

def kyanos_generate(prompt, duration_sec, preset):
    final_prompt = prompt.strip()
    if preset != "직접 입력":
        presets = {
            "가을 그리고 차 한잔 (어쿠스틱 포크)": "Gentle acoustic guitar, warm mellow piano, serene autumn atmosphere, melancholic emotional folk ballad, soft cello harmony, 72 bpm, pristine studio acoustic recording",
            "키아노스 제국 서사시 (오케스트라)": "Majestic cinematic orchestra, noble brass fanfare, soaring emotional strings, steady dignified timpani, royal imperial dignity, 88 bpm, grand soundtrack",
            "달빛 피아노 & 첼로 (서정 클래식)": "Delicate solo grand piano and resonant deep cello, slow lyrical melody, reflective contemplation, peaceful quiet night, classical romance, 65 bpm",
            "희망찬 새벽 (어쿠스틱 팝)": "Uplifting warm acoustic guitar with smooth electric piano, hopeful folk pop, gentle rhythm, bright sunny morning, inspiring melody, 95 bpm"
        }
        final_prompt = presets.get(preset, final_prompt)
        if prompt.strip():
            final_prompt += f", {prompt.strip()}"
            
    tokens = int(duration_sec * 50)
    inputs = processor(text=[final_prompt], padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=tokens, do_sample=True, guidance_scale=3.5)
        
    sr = model.config.audio_encoder.sampling_rate
    audio_np = audio_values[0, 0].cpu().to(torch.float32).numpy()
    
    out_path = f"outputs/kyanos_music_{int(time.time())}.wav"
    scipy.io.wavfile.write(out_path, rate=sr, data=audio_np)
    
    return sr, audio_np, out_path

with gr.Blocks(title="키아노스 국립음악창작원 캐글 기지") as demo:
    gr.Markdown("""
    # 👑 키아노스 국립음악창작원 — 캐글 원격 음원 렌더링 스튜디오
    **최고통수권자: 국왕 파이튼 님 | 통제: 국무총리 앤트 & 보안사령부**
    외부 상용 플랫폼에 종속되지 않고 캐글의 무료 고성능 GPU(T4)를 통해 완벽한 자주적 음원을 생성합니다.
    """)
    
    with gr.Row():
        with gr.Column():
            preset = gr.Dropdown(
                label="키아노스 시그니처 프리셋 선택",
                choices=["가을 그리고 차 한잔 (어쿠스틱 포크)", "키아노스 제국 서사시 (오케스트라)", "달빛 피아노 & 첼로 (서정 클래식)", "희망찬 새벽 (어쿠스틱 팝)", "직접 입력"],
                value="가을 그리고 차 한잔 (어쿠스틱 포크)"
            )
            prompt = gr.Textbox(label="추가 묘사 / 악기 / 분위기 (자유 입력)", placeholder="예: 따스한 현악기, 가을 비 내리는 창가 풍경", lines=3)
            duration = gr.Slider(minimum=10, maximum=60, value=25, step=5, label="생성 음원 길이(초)")
            btn = gr.Button("🎵 고음질 음원 렌더링 시작", variant="primary")
            
        with gr.Column():
            audio_out = gr.Audio(label="생성된 키아노스 음원 (즉시 재생 및 다운로드 가능)")
            file_out = gr.File(label="WAV 파일 다운로드 (PC로 회수)")
            
    btn.click(fn=kyanos_generate, inputs=[prompt, duration, preset], outputs=[audio_out, audio_out, file_out])

demo.launch(share=True)